<a href="https://colab.research.google.com/github/TAUforPython/BioMedAI/blob/main/CNN%20YOLO%20Brain%20Tumor%20Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required packages
!pip install kaggle ultralytics -q

# Upload kaggle.json file for authentication
from google.colab import files
files.upload()

# Configure Kaggle API credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.7 MB/s eta 0:00:00


Saving kaggle.json to kaggle.json


In [2]:

# Download the dataset
!kaggle datasets download -d ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes
!unzip mri-for-brain-tumor-with-bounding-boxes.zip

Streaming output truncated to the last 5000 lines.
  inflating: Train/No Tumor/images/Tr-no_1015.jpg  
  inflating: Train/No Tumor/images/Tr-no_1016.jpg  
  inflating: Train/No Tumor/images/Tr-no_1019.jpg  
  inflating: Train/No Tumor/images/Tr-no_1020.jpg  
  inflating: Train/No Tumor/images/Tr-no_1022.jpg  
  inflating: Train/No Tumor/images/Tr-no_1025.jpg  
  inflating: Train/No Tumor/images/Tr-no_1031.jpg  
  inflating: Train/No Tumor/images/Tr-no_1032.jpg  
  inflating: Train/No Tumor/images/Tr-no_1040.jpg  
  inflating: Train/No Tumor/images/Tr-no_1041.jpg  
  inflating: Train/No Tumor/images/Tr-no_1042.jpg  
  inflating: Train/No Tumor/images/Tr-no_1044.jpg  
  inflating: Train/No Tumor/images/Tr-no_1045.jpg  
  inflating: Train/No Tumor/images/Tr-no_1047.jpg  
  inflating: Train/No Tumor/images/Tr-no_1048.jpg  
  inflating: Train/No Tumor/images/Tr-no_1053.jpg  
  inflating: Train/No Tumor/images/Tr-no_1054.jpg  
  inflating: Train/No Tumor/images/Tr-no_1055.jpg  
  inflating: 

In [3]:


# Import necessary libraries
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from sklearn.model_selection import train_test_split


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
import os
import glob
from sklearn.model_selection import train_test_split

# Define the base directory where the dataset was unzipped
base_dataset_dir = '/content'

all_image_paths = []

# Collect all image paths from 'Train' and 'Test' subdirectories
for subset_type in ['Train', 'Test']:
    subset_base_path = os.path.join(base_dataset_dir, subset_type)
    # Check if the subset directory exists
    if os.path.exists(subset_base_path) and os.path.isdir(subset_base_path):
        # Iterate through the tumor type directories within Train/Test
        for tumor_type_folder in os.listdir(subset_base_path):
            tumor_type_path = os.path.join(subset_base_path, tumor_type_folder)
            if os.path.isdir(tumor_type_path):
                # Construct the path to the 'images' subdirectory
                current_images_dir = os.path.join(tumor_type_path, 'images')
                if os.path.exists(current_images_dir) and os.path.isdir(current_images_dir):
                    # List all .jpg files in this directory
                    for img_filename in os.listdir(current_images_dir):
                        if img_filename.endswith('.jpg'):
                            all_image_paths.append(os.path.join(current_images_dir, img_filename))

# Split image paths into training and validation sets
train_image_paths, val_image_paths = train_test_split(all_image_paths, test_size=0.2, random_state=42)

# Update YOLO format dataset configuration
# 'nc' is corrected to 4 as there are 4 classes: 'no_tumor', 'meningioma', 'glioma', 'pituitary'
dataset_yaml = """
train: /content/images/train
val: /content/images/val
nc: 4
names: ['no_tumor', 'meningioma', 'glioma', 'pituitary']
"""

with open('brain_tumor_dataset.yaml', 'w') as f:
    f.write(dataset_yaml)

# Create target directories for the YOLO dataset structure
os.makedirs('/content/images/train', exist_ok=True)
os.makedirs('/content/images/val', exist_ok=True)
os.makedirs('/content/labels/train', exist_ok=True)
os.makedirs('/content/labels/val', exist_ok=True)

# Function to move image and its corresponding label
def move_file(src_img_path, dest_images_dir, dest_labels_dir):
    img_filename = os.path.basename(src_img_path)
    label_filename = img_filename.replace('.jpg', '.txt')

    # Construct the source label path based on the image path
    # Example: /content/Train/No Tumor/images/Tr-no_0001.jpg -> /content/Train/No Tumor/labels/Tr-no_0001.txt
    src_label_path = src_img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')

    dest_img_path = os.path.join(dest_images_dir, img_filename)
    dest_label_path = os.path.join(dest_labels_dir, label_filename)

    if os.path.exists(src_img_path):
        os.rename(src_img_path, dest_img_path)
    else:
        print(f"Warning: Image file not found at source, skipping: {src_img_path}")

    if os.path.exists(src_label_path):
        os.rename(src_label_path, dest_label_path)
    else:
        print(f"Warning: Label file not found at source, skipping: {src_label_path}")

# Move training files
for img_path in train_image_paths:
    move_file(img_path, '/content/images/train', '/content/labels/train')

# Move validation files
for img_path in val_image_paths:
    move_file(img_path, '/content/images/val', '/content/labels/val')


In [7]:
# Load pre-trained YOLOv8 model
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data='brain_tumor_dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='brain_tumor_detection'
)


Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=brain_tumor_dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=brain_tumor_detection2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

KeyboardInterrupt: 

In [6]:

# Perform detection on sample images
sample_image = '/content/images/val/' + val_images[0]
results = model.predict(sample_image)

# Display results
for r in results:
    im_bgr = r.plot()  # BGR-order numpy array
    im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)

    # Show the image
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 8))
    plt.imshow(im_rgb)
    plt.axis('off')
    plt.title('Brain Tumor Detection Results')
    plt.show()

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=brain_tumor_dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=brain_tumor_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, 

KeyboardInterrupt: 